# Ethena Loop on Aave V3 — Growth Assignment

**Role:** Protocol Growth, Aave Labs · **Market:** Aave V3 Ethereum Core · **Assets:** USDe (debt) / sUSDe (collateral)

Reproducible working behind the 2-page memo — every number traces to a cell here. Parameters are read live from Aave V3 contracts via JSON-RPC where possible; assumptions that can't be sourced on-chain are flagged inline.

## Section 0 — Data sourcing

Addresses from `bgd-labs/aave-address-book` (main branch). Reserve/rate/E-mode figures read live via `eth_call`. USDe/sUSDe supply read directly from the ERC-20 contracts. Ethena's yield split isn't on-chain — sourced from their app API and cited separately with its own timestamp.

In [9]:
from web3 import Web3
import datetime, json

# Archive RPC required -- a pinned historical block needs an endpoint that retains
# that state. publicnode.com prunes after ~1hr and rejected this block; drpc.eth
# verified to retain state back to block 18,000,000 before adopting it.
RPC = "https://eth.drpc.org"
# fallback if drpc has an outage: "https://eth-mainnet.public.blastapi.io"
w3 = Web3(Web3.HTTPProvider(RPC))
assert w3.is_connected(), "RPC not reachable"

# --- Canonical addresses, source: bgd-labs/aave-address-book, AaveV3Ethereum.sol (main branch) ---
POOL              = w3.to_checksum_address("0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2")
POOL_ADDRESSES_PROVIDER = w3.to_checksum_address("0x2f39d218133AFaB8F2B819B1066c7E434Ad94E9e")
DATA_PROVIDER     = w3.to_checksum_address("0x0a16f2FCC0D44FaE41cc54e079281D84A363bECD")
USDe              = w3.to_checksum_address("0x4c9EDD5852cd905f086C759E8383e09bff1E68B3")
sUSDe             = w3.to_checksum_address("0x9D39A5DE30e57443BfF2A8307A4256c8797A3497")
USDe_IR_STRATEGY  = w3.to_checksum_address("0x9ec6F08190DeA04A54f8Afc53Db96134e5E3FdFB")

# Sibling Ethereum V3 deployments, for the market-identity check below
LIDO_PROVIDER    = w3.to_checksum_address("0xcfBf336fe147D643B9Cb705648500e101504B16d")
ETHERFI_PROVIDER = w3.to_checksum_address("0xeBa440B438Ad808101d1c451C1C5322c90BEFCdA")

# Pinned to a fixed block (not "latest") so every rerun reads identical on-chain state.
BLOCK_NUMBER = 25682519
BLOCK_TS = datetime.datetime.utcfromtimestamp(1785858143)
print(f"Reading at pinned block {BLOCK_NUMBER}, {BLOCK_TS} UTC via {RPC}")

Reading at pinned block 25682519, 2026-08-04 15:42:23 UTC via https://eth.drpc.org


In [10]:
# Minimal ABIs — just the functions we need
DATA_PROVIDER_ABI = json.loads("""[
 {"name":"getReserveConfigurationData","type":"function","stateMutability":"view",
  "inputs":[{"name":"asset","type":"address"}],
  "outputs":[{"name":"decimals","type":"uint256"},{"name":"ltv","type":"uint256"},
   {"name":"liquidationThreshold","type":"uint256"},{"name":"liquidationBonus","type":"uint256"},
   {"name":"reserveFactor","type":"uint256"},{"name":"usageAsCollateralEnabled","type":"bool"},
   {"name":"borrowingEnabled","type":"bool"},{"name":"stableBorrowRateEnabled","type":"bool"},
   {"name":"isActive","type":"bool"},{"name":"isFrozen","type":"bool"}]},
 {"name":"getReserveCaps","type":"function","stateMutability":"view",
  "inputs":[{"name":"asset","type":"address"}],
  "outputs":[{"name":"borrowCap","type":"uint256"},{"name":"supplyCap","type":"uint256"}]},
 {"name":"getReserveData","type":"function","stateMutability":"view",
  "inputs":[{"name":"asset","type":"address"}],
  "outputs":[{"name":"unbacked","type":"uint256"},{"name":"accruedToTreasuryScaled","type":"uint256"},
   {"name":"totalAToken","type":"uint256"},{"name":"totalStableDebt","type":"uint256"},
   {"name":"totalVariableDebt","type":"uint256"},{"name":"liquidityRate","type":"uint256"},
   {"name":"variableBorrowRate","type":"uint256"},{"name":"stableBorrowRate","type":"uint256"},
   {"name":"averageStableBorrowRate","type":"uint256"},{"name":"liquidityIndex","type":"uint256"},
   {"name":"variableBorrowIndex","type":"uint256"},{"name":"lastUpdateTimestamp","type":"uint40"}]}
]""")

IR_STRATEGY_ABI = json.loads("""[
 {"name":"getInterestRateDataBps","type":"function","stateMutability":"view",
  "inputs":[{"name":"reserve","type":"address"}],
  "outputs":[{"name":"","type":"tuple","components":[
    {"name":"optimalUsageRatio","type":"uint16"},
    {"name":"baseVariableBorrowRate","type":"uint32"},
    {"name":"variableRateSlope1","type":"uint32"},
    {"name":"variableRateSlope2","type":"uint32"}]}]}
]""")

POOL_ABI = json.loads("""[
 {"name":"getEModeCategoryData","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],
  "outputs":[{"name":"","type":"tuple","components":[
    {"name":"ltv","type":"uint16"},{"name":"liquidationThreshold","type":"uint16"},
    {"name":"liquidationBonus","type":"uint16"},{"name":"priceSource","type":"address"},
    {"name":"label","type":"string"}]}]},
 {"name":"getReservesList","type":"function","stateMutability":"view","inputs":[],
  "outputs":[{"type":"address[]"}]},
 {"name":"ADDRESSES_PROVIDER","type":"function","stateMutability":"view","inputs":[],
  "outputs":[{"type":"address"}]},
 {"name":"getEModeCategoryCollateralBitmap","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],"outputs":[{"type":"uint128"}]},
 {"name":"getEModeCategoryBorrowableBitmap","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],"outputs":[{"type":"uint128"}]},
 {"name":"getEModeCategoryLabel","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],"outputs":[{"type":"string"}]},
 {"name":"getEModeCategoryCollateralConfig","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],
  "outputs":[{"type":"tuple","components":[{"name":"ltv","type":"uint16"},
   {"name":"liquidationThreshold","type":"uint16"},{"name":"liquidationBonus","type":"uint16"}]}]}
]""")

ADDRESSES_PROVIDER_ABI = json.loads("""[
 {"name":"getMarketId","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"string"}]},
 {"name":"getPool","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"address"}]}
]""")

ERC20_ABI = json.loads("""[
 {"name":"totalSupply","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"uint256"}]},
 {"name":"balanceOf","type":"function","stateMutability":"view","inputs":[{"name":"a","type":"address"}],"outputs":[{"type":"uint256"}]}
]""")

data_provider = w3.eth.contract(address=DATA_PROVIDER, abi=DATA_PROVIDER_ABI)
ir_strategy   = w3.eth.contract(address=USDe_IR_STRATEGY, abi=IR_STRATEGY_ABI)
pool          = w3.eth.contract(address=POOL, abi=POOL_ABI)
addresses_provider = w3.eth.contract(address=POOL_ADDRESSES_PROVIDER, abi=ADDRESSES_PROVIDER_ABI)
lido_provider       = w3.eth.contract(address=LIDO_PROVIDER, abi=ADDRESSES_PROVIDER_ABI)
etherfi_provider    = w3.eth.contract(address=ETHERFI_PROVIDER, abi=ADDRESSES_PROVIDER_ABI)
usde_token    = w3.eth.contract(address=USDe, abi=ERC20_ABI)
susde_token   = w3.eth.contract(address=sUSDe, abi=ERC20_ABI)

print("Contracts wired.")

Contracts wired.


## Section 0.1 — Verify we're looking at the right market and the right tokens

Two checks, both reproducible manually on Etherscan:

**(a) Market identity.** Aave has three Ethereum V3 deployments (Core, Prime/Lido, EtherFi), each with its own `PoolAddressesProvider` and `getMarketId()`. Diff all three below rather than trust the address-book's file naming.

**(b) Reserve identity.** `getReservesList()` returns every reserve in a fixed order; hardcode USDe's index (30) and sUSDe's (32) rather than string-matching addresses, so it's checkable by counting to a position in Etherscan's returned array.

In [11]:
# --- (a) Market identity: diff the marketId string across all three Ethereum V3 deployments ---
market_ids = {
    "Core (ours)": addresses_provider.functions.getMarketId().call(block_identifier=BLOCK_NUMBER),
    "Prime/Lido":  lido_provider.functions.getMarketId().call(block_identifier=BLOCK_NUMBER),
    "EtherFi":     etherfi_provider.functions.getMarketId().call(block_identifier=BLOCK_NUMBER),
}
for name, mid in market_ids.items():
    print(f"{name:14s} -> {mid!r}")

# Mutual pointer check: Pool -> Provider and Provider -> Pool must agree
pool_to_provider = pool.functions.ADDRESSES_PROVIDER().call(block_identifier=BLOCK_NUMBER)
provider_to_pool = addresses_provider.functions.getPool().call(block_identifier=BLOCK_NUMBER)
assert pool_to_provider == POOL_ADDRESSES_PROVIDER, "Pool does not point back to our provider"
assert provider_to_pool == POOL, "Provider does not point back to our Pool"
print(f"\nMutual pointer check passed: Pool <-> PoolAddressesProvider agree in both directions.")

Core (ours)    -> 'Aave Ethereum Market'
Prime/Lido     -> 'Aave V3 Lido Ethereum'
EtherFi        -> 'Aave V3 EtherFi Ethereum'

Mutual pointer check passed: Pool <-> PoolAddressesProvider agree in both directions.


In [12]:
# --- (b) Reserve identity: hardcoded index into getReservesList(), verified at our pinned block ---
USDe_RESERVE_INDEX  = 30
sUSDe_RESERVE_INDEX = 32

reserves_list = pool.functions.getReservesList().call(block_identifier=BLOCK_NUMBER)
print(f"Total reserves in this Pool at block {BLOCK_NUMBER}: {len(reserves_list)}")

assert reserves_list[USDe_RESERVE_INDEX] == USDe, \
    f"index {USDe_RESERVE_INDEX} is {reserves_list[USDe_RESERVE_INDEX]}, expected USDe {USDe}"
assert reserves_list[sUSDe_RESERVE_INDEX] == sUSDe, \
    f"index {sUSDe_RESERVE_INDEX} is {reserves_list[sUSDe_RESERVE_INDEX]}, expected sUSDe {sUSDe}"

print(f"reserves[{USDe_RESERVE_INDEX}]  = {reserves_list[USDe_RESERVE_INDEX]}  == USDe   OK")
print(f"reserves[{sUSDe_RESERVE_INDEX}] = {reserves_list[sUSDe_RESERVE_INDEX]}  == sUSDe  OK")
print("\nManual cross-check: call getReservesList on Etherscan's Read as Proxy tab for")
print("0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2, count to element 30 (0-indexed) and")
print("element 32, and compare against the two addresses printed above.")

Total reserves in this Pool at block 25682519: 67
reserves[30]  = 0x4c9EDD5852cd905f086C759E8383e09bff1E68B3  == USDe   OK
reserves[32] = 0x9D39A5DE30e57443BfF2A8307A4256c8797A3497  == sUSDe  OK

Manual cross-check: call getReservesList on Etherscan's Read as Proxy tab for
0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2, count to element 30 (0-indexed) and
element 32, and compare against the two addresses printed above.


In [13]:
# --- USDe reserve ---
usde_cfg   = data_provider.functions.getReserveConfigurationData(USDe).call(block_identifier=BLOCK_NUMBER)
usde_caps  = data_provider.functions.getReserveCaps(USDe).call(block_identifier=BLOCK_NUMBER)
usde_data  = data_provider.functions.getReserveData(USDe).call(block_identifier=BLOCK_NUMBER)
usde_ir    = ir_strategy.functions.getInterestRateDataBps(USDe).call(block_identifier=BLOCK_NUMBER)

# --- sUSDe reserve ---
susde_cfg  = data_provider.functions.getReserveConfigurationData(sUSDe).call(block_identifier=BLOCK_NUMBER)
susde_caps = data_provider.functions.getReserveCaps(sUSDe).call(block_identifier=BLOCK_NUMBER)

RAY = 10**27
WAD = 10**18

usde = {
    "decimals": usde_cfg[0], "ltv_standalone_pct": usde_cfg[1]/100, "lt_standalone_pct": usde_cfg[2]/100,
    "liq_bonus_pct": (usde_cfg[3]-10000)/100, "reserve_factor_pct": usde_cfg[4]/100,
    "collateral_enabled": usde_cfg[5], "borrowing_enabled": usde_cfg[6], "is_active": usde_cfg[8], "is_frozen": usde_cfg[9],
    "borrow_cap": usde_caps[0], "supply_cap": usde_caps[1],
    "total_supplied": usde_data[2]/WAD, "total_variable_debt": usde_data[4]/WAD,
    "liquidity_rate_pct": usde_data[5]/RAY*100, "variable_borrow_rate_pct": usde_data[6]/RAY*100,
    "last_update": datetime.datetime.utcfromtimestamp(usde_data[11]),
    # NB: getInterestRateDataBps returns basis points (1% = 100 bps) for all four fields
    "uopt_pct": usde_ir[0]/100, "base_rate_pct": usde_ir[1]/100, "slope1_pct": usde_ir[2]/100, "slope2_pct": usde_ir[3]/100,
}
usde["utilization_pct"] = usde["total_variable_debt"] / usde["total_supplied"] * 100

susde = {
    "decimals": susde_cfg[0], "ltv_standalone_pct": susde_cfg[1]/100, "lt_standalone_pct": susde_cfg[2]/100,
    "liq_bonus_pct": (susde_cfg[3]-10000)/100, "reserve_factor_pct": susde_cfg[4]/100,
    "collateral_enabled": susde_cfg[5], "borrowing_enabled": susde_cfg[6], "is_active": susde_cfg[8],
    "supply_cap": susde_caps[1],
}

# sanity check: does the published borrow-rate formula reproduce the on-chain variableBorrowRate?
U = usde["utilization_pct"]
if U <= usde["uopt_pct"]:
    implied_rate = usde["base_rate_pct"] + usde["slope1_pct"] * (U / usde["uopt_pct"])
else:
    implied_rate = usde["base_rate_pct"] + usde["slope1_pct"] + usde["slope2_pct"] * ((U - usde["uopt_pct"]) / (100 - usde["uopt_pct"]))
usde["implied_borrow_rate_check_pct"] = implied_rate

print("USDe :", json.dumps({k:(str(v) if isinstance(v, datetime.datetime) else v) for k,v in usde.items()}, indent=2))
print("sUSDe:", json.dumps(susde, indent=2))

USDe : {
  "decimals": 18,
  "ltv_standalone_pct": 0.0,
  "lt_standalone_pct": 75.0,
  "liq_bonus_pct": 8.5,
  "reserve_factor_pct": 25.0,
  "collateral_enabled": true,
  "borrowing_enabled": true,
  "is_active": true,
  "is_frozen": false,
  "borrow_cap": 700000000,
  "supply_cap": 902000000,
  "total_supplied": 785161563.4978043,
  "total_variable_debt": 582621784.7580878,
  "liquidity_rate_pct": 1.8352717971763681,
  "variable_borrow_rate_pct": 3.2978303294547096,
  "last_update": "2026-08-04 15:42:11",
  "uopt_pct": 90.0,
  "base_rate_pct": 0.0,
  "slope1_pct": 4.0,
  "slope2_pct": 12.0,
  "utilization_pct": 74.20406344938421,
  "implied_borrow_rate_check_pct": 3.297958375528187
}
sUSDe: {
  "decimals": 18,
  "ltv_standalone_pct": 0.0,
  "lt_standalone_pct": 75.0,
  "liq_bonus_pct": 8.5,
  "reserve_factor_pct": 20.0,
  "collateral_enabled": true,
  "borrowing_enabled": false,
  "is_active": true,
  "supply_cap": 450000000
}


### E-mode category selection

Every category's collateral and borrowable bitmaps are scanned directly (not the address-book's label) to find categories where sUSDe is collateral and USDe is borrowable. Some pair sUSDe with a Pendle PT written on a third-party tranching protocol ("Strata") rather than sUSDe itself — not the pure USDe/sUSDe pairing this assignment is about.

**Test:** for each candidate category, resolve every non-sUSDe collateral bit's Pendle PT → `SY()` → `yieldToken()`, and compare that address to sUSDe directly (not a label match). Categories 39/45/47/48 route through Strata; only 17/24/31/32/36/37 are direct. Among the direct-only set, **category 32** has the best terms: LTV 92.00%, LT 94.00%, bonus 2.00% — same tier as 39, just without the Strata detour.

In [14]:
import pandas as pd

# --- Step 1: scan every category, check both bitmaps directly, don't trust labels ---
USDe_RESERVE_INDEX_ = USDe_RESERVE_INDEX   # 30
sUSDe_RESERVE_INDEX_ = sUSDe_RESERVE_INDEX  # 32

reserves_list = pool.functions.getReservesList().call(block_identifier=BLOCK_NUMBER)

candidates = []
highest_id_seen = -1
for cid in range(0, 100):
    try:
        coll_bm = pool.functions.getEModeCategoryCollateralBitmap(cid).call(block_identifier=BLOCK_NUMBER)
        borrow_bm = pool.functions.getEModeCategoryBorrowableBitmap(cid).call(block_identifier=BLOCK_NUMBER)
    except Exception:
        continue
    if coll_bm != 0 or borrow_bm != 0:
        highest_id_seen = cid
    susde_collateral = bool(coll_bm & (1 << sUSDe_RESERVE_INDEX_))
    usde_borrowable  = bool(borrow_bm & (1 << USDe_RESERVE_INDEX_))
    if susde_collateral and usde_borrowable:
        label = pool.functions.getEModeCategoryLabel(cid).call(block_identifier=BLOCK_NUMBER)
        cfg = pool.functions.getEModeCategoryCollateralConfig(cid).call(block_identifier=BLOCK_NUMBER)
        candidates.append({
            "category_id": cid, "label": label, "collateral_bitmap": coll_bm,
            "ltv_pct": cfg[0]/100, "lt_pct": cfg[1]/100, "liq_bonus_pct": (cfg[2]-10000)/100,
        })

print(f"Highest category id with a non-empty bitmap found in range 0-99: {highest_id_seen}")
if highest_id_seen >= 90:
    print("WARNING: highest configured id is close to the scan's upper bound -- widen the range further.")

# --- Step 2: classify OTHER collateral assets, address-based not label-based ---
# Resolve each non-sUSDe bit's Pendle PT -> SY() -> yieldToken() and compare to sUSDe's
# own address. "pure_ethena" only if every other asset resolves back to sUSDe itself.
PT_ABI = json.loads("""[
 {"name":"SY","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"address"}]}
]""")
SY_ABI = json.loads("""[
 {"name":"yieldToken","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"address"}]}
]""")

for c in candidates:
    other_bits = [i for i in range(len(reserves_list))
                  if (c["collateral_bitmap"] & (1 << i)) and i != sUSDe_RESERVE_INDEX_]
    is_pure = True
    for i in other_bits:
        try:
            pt = w3.eth.contract(address=reserves_list[i], abi=PT_ABI)
            sy_addr = pt.functions.SY().call(block_identifier=BLOCK_NUMBER)
            sy = w3.eth.contract(address=sy_addr, abi=SY_ABI)
            underlying = sy.functions.yieldToken().call(block_identifier=BLOCK_NUMBER)
            if underlying != sUSDe:
                is_pure = False
        except Exception:
            is_pure = False  # can't verify -> don't trust it
    c["pure_ethena_susde"] = is_pure

cand_df = pd.DataFrame(candidates).drop(columns=["collateral_bitmap"]) \
    .sort_values(["ltv_pct", "liq_bonus_pct"], ascending=[False, True]).reset_index(drop=True)
display(cand_df)

pure_df = cand_df[cand_df["pure_ethena_susde"]].reset_index(drop=True)
emode_max = pure_df.iloc[0].to_dict()             # best category with NO third-party token involved
emode_current = cand_df[cand_df["category_id"] == 48].iloc[0].to_dict()  # kept for reference, Strata-routed
emode_data = emode_max  # kept for backward compatibility with downstream cells

print(f"\nBest pure sUSDe/USDe category (no third-party tranching): category {emode_max['category_id']} ({emode_max['label']}), LTV {emode_max['ltv_pct']}%")
print(f"For reference, category 48 (Strata-routed, previously used as 'current'): LTV {emode_current['ltv_pct']}%")

lev = 1 / (1 - emode_max["ltv_pct"]/100)
print(f"\nMax leverage on the pure sUSDe/USDe category: {lev:.2f}x")

Highest category id with a non-empty bitmap found in range 0-99: 48


,category_id,label,ltv_pct,lt_pct,liq_bonus_pct,pure_ethena_susde
0,39,PT_srUSDe_2APR2026_sUSDe__USDe,92.00,94.00,1.30,False
1,32,PTsUSDe5FEB/USDe,92.00,94.00,2.00,True
2,37,PT_sUSDe_7MAY2026__USDe,92.00,94.00,3.20,True
3,45,PT_srUSDe_25JUN2026__USDe,91.60,93.60,1.30,False
4,48,PT-srUSDe USDe,91.26,93.26,2.18,False
5,31,PTsUSDe5FEB/Stablecoins,90.00,92.00,3.00,True
6,17,PT-sUSDe Stablecoins September 2025,90.00,92.00,3.10,True
7,24,PT-sUSDe Stablecoins Nov 2025,90.00,92.00,3.10,True
8,38,PT_srUSDe_2APR2026_sUSDe__USDT_USDe_USDC,90.00,92.00,3.30,False
9,44,PT_srUSDe_25JUN2026__Stablecoins,90.00,92.00,4.10,False



Best pure sUSDe/USDe category (no third-party tranching): category 32 (PTsUSDe5FEB/USDe), LTV 92.0%
For reference, category 48 (Strata-routed, previously used as 'current'): LTV 91.26%

Max leverage on the pure sUSDe/USDe category: 12.50x


### Illustrating the two families

Decode the collateral basket for the winning category (32) and the rejected Strata-routed one (39), token-by-token, to make the `pure_ethena_susde` classification checkable.

In [15]:
SYMBOL_ABI = json.loads("""[
 {"name":"symbol","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"string"}]},
 {"name":"name","type":"function","stateMutability":"view","inputs":[],"outputs":[{"type":"string"}]}
]""")

def decode_basket(category_id):
    coll_bm = pool.functions.getEModeCategoryCollateralBitmap(category_id).call(block_identifier=BLOCK_NUMBER)
    set_bits = [i for i in range(len(reserves_list)) if coll_bm & (1 << i)]
    print(f"Category {category_id} collateral bitmap = {coll_bm}, set bit indices = {set_bits}")
    for i in set_bits:
        addr = reserves_list[i]
        tok = w3.eth.contract(address=addr, abi=SYMBOL_ABI)
        sym = tok.functions.symbol().call(block_identifier=BLOCK_NUMBER)
        name = tok.functions.name().call(block_identifier=BLOCK_NUMBER)
        line = f"  index {i}: {addr}  symbol={sym!r}  name={name!r}"
        if sym.startswith("PT-"):
            pt = w3.eth.contract(address=addr, abi=PT_ABI)
            sy_addr = pt.functions.SY().call(block_identifier=BLOCK_NUMBER)
            sy = w3.eth.contract(address=sy_addr, abi=SY_ABI)
            underlying = sy.functions.yieldToken().call(block_identifier=BLOCK_NUMBER)
            line += f"\n    -> SY.yieldToken() = {underlying}  == sUSDe? {underlying == sUSDe}"
        print(line)
    print()

print("=== Winning category: 32 (pure sUSDe/USDe, no third-party token) ===")
decode_basket(32)
print("=== Rejected category: 39 (routes through Strata's tranching protocol) ===")
decode_basket(39)

=== Winning category: 32 (pure sUSDe/USDe, no third-party token) ===
Category 32 collateral bitmap = 81064797587636224, set bit indices = [32, 53, 56]
  index 32: 0x9D39A5DE30e57443BfF2A8307A4256c8797A3497  symbol='sUSDe'  name='Staked USDe'
  index 53: 0xe6A934089BBEe34F832060CE98848359883749B3  symbol='PT-sUSDE-27NOV2025'  name='PT Ethena sUSDE 27NOV2025'
    -> SY.yieldToken() = 0x9D39A5DE30e57443BfF2A8307A4256c8797A3497  == sUSDe? True
  index 56: 0xE8483517077afa11A9B07f849cee2552f040d7b2  symbol='PT-sUSDE-5FEB2026'  name='PT Ethena sUSDE 5FEB2026'
    -> SY.yieldToken() = 0x9D39A5DE30e57443BfF2A8307A4256c8797A3497  == sUSDe? True

=== Rejected category: 39 (routes through Strata's tranching protocol) ===
Category 39 collateral bitmap = 4611686022722355200, set bit indices = [32, 62]
  index 32: 0x9D39A5DE30e57443BfF2A8307A4256c8797A3497  symbol='sUSDe'  name='Staked USDe'
  index 62: 0x9Bf45ab47747F4B4dD09B3C2c73953484b4eB375  symbol='PT-srUSDe-2APR2026'  name='PT Strata Senior U

In [16]:
# --- USDe / sUSDe supply and staking ratio, read directly from the token contracts ---
usde_total_supply = usde_token.functions.totalSupply().call(block_identifier=BLOCK_NUMBER) / WAD
usde_staked        = usde_token.functions.balanceOf(sUSDe).call(block_identifier=BLOCK_NUMBER) / WAD   # USDe locked in the sUSDe staking contract
susde_shares       = susde_token.functions.totalSupply().call(block_identifier=BLOCK_NUMBER) / WAD

ethena = {
    "usde_total_supply": usde_total_supply,
    "usde_staked_in_susde": usde_staked,
    "staking_ratio_pct": usde_staked / usde_total_supply * 100,
    "susde_shares_outstanding": susde_shares,
    "susde_exchange_rate": usde_staked / susde_shares,  # USDe per sUSDe share
}
print(json.dumps(ethena, indent=2))

{
  "usde_total_supply": 3875645933.090682,
  "usde_staked_in_susde": 1559254171.9393601,
  "staking_ratio_pct": 40.23211095281641,
  "susde_shares_outstanding": 1255416775.0915062,
  "susde_exchange_rate": 1.2420211382197817
}


### Ethena yield split and incentives — not on-chain, cited separately

- **Source:** Ethena app yield API, read 2026-08-04 (data as of 2026-07-29): USDe protocol yield **4.57%**, sUSDe net yield **4.00%**. The ~57bp gap is Ethena's take (Insurance/Reserve Fund, plus, since Q1 2026, a share to sENA stakers) — no on-chain breakdown, so treated as a flagged ~12%-of-gross assumption where needed.
- **Incentives:** Ethena's Merkl "Liquid Leverage" campaign shows 0 active campaigns as of 2026-08-04 (9 past, most recent inactive) — assume no active Aave-side incentive on top of the base spread; re-verify at submission time since these turn on and off.

In [17]:
# --- Consolidated parameter table (this is the "Setup" table for the memo) ---
susde_gross_yield_pct = 4.57
susde_net_yield_pct   = 4.00

PARAMS = {
    "block_number": BLOCK_NUMBER, "block_time_utc": str(BLOCK_TS),
    "usde_borrow_apy_pct": round(usde["variable_borrow_rate_pct"], 3),
    "usde_supply_apy_pct": round(usde["liquidity_rate_pct"], 3),
    "susde_net_yield_pct": susde_net_yield_pct,
    "susde_gross_protocol_yield_pct": susde_gross_yield_pct,
    "usde_reserve_factor_pct": usde["reserve_factor_pct"],
    "susde_reserve_factor_pct": susde["reserve_factor_pct"],
    "ir_base_pct": usde["base_rate_pct"], "ir_uopt_pct": usde["uopt_pct"],
    "ir_slope1_pct": usde["slope1_pct"], "ir_slope2_pct": usde["slope2_pct"],
    "usde_supply_cap": usde["supply_cap"], "usde_borrow_cap": usde["borrow_cap"],
    "susde_supply_cap": susde["supply_cap"],
    "usde_total_supplied": round(usde["total_supplied"]), "usde_total_borrowed": round(usde["total_variable_debt"]),
    "usde_utilization_pct": round(usde["utilization_pct"], 2),
    "emode_pure_id": emode_max["category_id"], "emode_pure_label": emode_max["label"],
    "emode_pure_ltv_pct": emode_max["ltv_pct"], "emode_pure_lt_pct": emode_max["lt_pct"],
    "emode_pure_liq_bonus_pct": emode_max["liq_bonus_pct"],
    "emode_strata_reference_id": emode_current["category_id"], "emode_strata_reference_label": emode_current["label"],
    "emode_strata_reference_ltv_pct": emode_current["ltv_pct"], "emode_strata_reference_lt_pct": emode_current["lt_pct"],
    "emode_strata_reference_liq_bonus_pct": emode_current["liq_bonus_pct"],
    "usde_total_supply_ethena": round(ethena["usde_total_supply"]),
    "usde_staking_ratio_pct": round(ethena["staking_ratio_pct"], 2),
    "active_aave_incentive": "none observed (Merkl Liquid Leverage campaign shows 0 active as of 2026-08-04)",
}

pd.set_option("display.max_rows", None)
display(pd.Series(PARAMS, name="value").to_frame())

,value
block_number,25682519
block_time_utc,2026-08-04 15:42:23
usde_borrow_apy_pct,3.298
usde_supply_apy_pct,1.835
susde_net_yield_pct,4.0
susde_gross_protocol_yield_pct,4.57
usde_reserve_factor_pct,25.0
susde_reserve_factor_pct,20.0
ir_base_pct,0.0
ir_uopt_pct,90.0
